# <a id='toc1_'></a>[How adherent is the medial literature to HGNC recommendations on the use of gene identifiers?](#toc0_)

This analysis assesses how closely the medical literature follows HGNC recommendations for unambiguous gene identification. Using PubTator 3 annotations, the code identifies papers(titles, abstracts, or full text when available) in which an alias is tagged as a gene and determines whether the mention is linked to an HGNC, Ensembl, or NCBI Gene identifier. The resulting proportion provides a measure of identifier use for an alias symbol in published literature.

**Table of contents**<a id='toc0_'></a>    
- [How adherent is the medial literature to HGNC recommendations on the use of gene identifiers?](#toc1_)    
  - [Example Gene Pairs](#toc1_1_)    
  - [Download cache for ERBB and ASP gene pairs](#toc1_2_)    
  - [Download results for ACMG gene set](#toc1_3_)    
  - [Results](#toc1_4_)    
    - [Ambiguous symbols](#toc1_4_1_)    
    - [ACMG gene set](#toc1_4_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import shutil
from collections import defaultdict
from pathlib import Path
import pickle
from requests.exceptions import HTTPError
import json
import pandas as pd
import functions as fn
from acmg_gene_set import GENES
import json
from dataclasses import asdict

from collections import Counter


## <a id='toc1_1_'></a>[Example Gene Pairs](#toc0_)

In [3]:
ERBB_EGFR = fn.GenePair(
    alias="ERBB",
    approved_symbol="EGFR",
    ncbi_gene_id="1956",
    hgnc_id="HGNC:3236",
    ensembl_gene_id="ENSG00000146648",
)

ASP_TMPRSS11D = fn.GenePair(
    alias="ASP",
    approved_symbol="TMPRSS11D",
    ncbi_gene_id="9407",
    hgnc_id="HGNC:24059",
    ensembl_gene_id="ENSG00000153802",
)

ASP_C3 = fn.GenePair(
    alias="ASP",
    approved_symbol="C3",
    ncbi_gene_id="718",
    hgnc_id="HGNC:1318",
    ensembl_gene_id="ENSG00000125730",
)

ASP_ASPM = fn.GenePair(
    alias="ASP",
    approved_symbol="ASPM",
    ncbi_gene_id="259266",
    hgnc_id="HGNC:19048",
    ensembl_gene_id="ENSG00000066279",
)

ASP_A1CF = fn.GenePair(
    alias="ASP",
    approved_symbol="A1CF",
    ncbi_gene_id="29974",
    hgnc_id="HGNC:24086",
    ensembl_gene_id="ENSG00000148584",
)

ASP_ASIP = fn.GenePair(
    alias="ASP",
    approved_symbol="ASIP",
    ncbi_gene_id="434",
    hgnc_id="HGNC:745",
    ensembl_gene_id="ENSG00000101440",
)

ASP_ROPN1L = fn.GenePair(
    alias="ASP",
    approved_symbol="ROPN1L",
    ncbi_gene_id="83853",
    hgnc_id="HGNC:24060",
    ensembl_gene_id="ENSG00000145491",
)

ASP_ATG5 = fn.GenePair(
    alias="ASP",
    approved_symbol="ATG5",
    ncbi_gene_id="9474",
    hgnc_id="HGNC:589",
    ensembl_gene_id="ENSG00000057663",
)

ASP_ASPA = fn.GenePair(
    alias="ASP",
    approved_symbol="ASPA",
    ncbi_gene_id="443",
    hgnc_id="HGNC:756",
    ensembl_gene_id="ENSG00000108381",
)


## <a id='toc1_2_'></a>[Download cache for ERBB and ASP gene pairs](#toc0_)

In [4]:
CACHE_URL = "https://nch-igm-wagner-lab-public.s3.us-east-2.amazonaws.com/genejar/output/"

In [5]:
ASP_URL = f"{CACHE_URL}asp"
ERBB_URL = f"{CACHE_URL}erbb"

In [6]:
cache_dir = Path("output/erbb/document_cache")
zip_path = Path("output/erbb/document_cache.zip")

if not cache_dir.exists() or not any(cache_dir.iterdir()):
    cache_dir.parent.mkdir(parents=True, exist_ok=True)

    fn.download_s3(
        f"{ERBB_URL}/document_cache.zip",
        zip_path,
    )

    shutil.unpack_archive(
        zip_path,
        cache_dir.parent,
    )

    zip_path.unlink()

In [7]:
cache_dir = Path("output/asp/document_cache")
zip_path = Path("output/asp/document_cache.zip")

if not cache_dir.exists() or not any(cache_dir.iterdir()):
    cache_dir.parent.mkdir(parents=True, exist_ok=True)

    fn.download_s3(
        f"{ASP_URL}/document_cache.zip",
        zip_path,
    )

    shutil.unpack_archive(
        zip_path,
        cache_dir.parent,
    )

    zip_path.unlink()

## <a id='toc1_3_'></a>[Download results for ACMG gene set](#toc0_)

In [ ]:
json_results = json.loads(
    Path("output/all_results.json").read_text()
)

all_results = {}

for symbol, result in json_results.items():
    all_results[symbol] = {
        "pair": fn.GenePair(**result["pair"]),
        "processed_documents": result["processed_documents"],
        "candidate_papers": result["candidate_papers"],
        "papers_with_alias_gene_annotation": set(
            result["papers_with_alias_gene_annotation"]
        ),
        "papers_with_identifier_in_text": set(
            result["papers_with_identifier_in_text"]
        ),
        "papers_by_namespace": {
            namespace: set(pmids)
            for namespace, pmids
            in result["papers_by_namespace"].items()
        },
        "annotation_identifier_counts": Counter(
            result["annotation_identifier_counts"]
        ),
        "papers_by_section": {
            section: set(pmids)
            for section, pmids
            in result["papers_by_section"].items()
        },
        "denominator": result["denominator"],
        "numerator": result["numerator"],
        "percentage": result["percentage"],
    }

In [ ]:
def save_all_results_json(all_results):
    """Save analysis results in JSON format."""

    json_results = {}

    for symbol, result in all_results.items():
        json_results[symbol] = {
            "pair": asdict(result["pair"]),
            "processed_documents":
                result["processed_documents"],
            "candidate_papers":
                result["candidate_papers"],
            "papers_with_alias_gene_annotation": sorted(
                result["papers_with_alias_gene_annotation"]
            ),
            "papers_with_identifier_in_text": sorted(
                result["papers_with_identifier_in_text"]
            ),
            "papers_by_namespace": {
                namespace: sorted(pmids)
                for namespace, pmids
                in result["papers_by_namespace"].items()
            },
            "annotation_identifier_counts": dict(
                result["annotation_identifier_counts"]
            ),
            "papers_by_section": {
                section: sorted(pmids)
                for section, pmids
                in result["papers_by_section"].items()
            },
            "denominator": result["denominator"],
            "numerator": result["numerator"],
            "percentage": result["percentage"],
        }

    Path("output/all_results.json").write_text(
        json.dumps(json_results, indent=2)
    )

In [33]:
for gene_pair in missing_gene_pairs:
    print(
        f"Analyzing {gene_pair.alias} -> "
        f"{gene_pair.approved_symbol}..."
    )

    candidate_pmids = fn.load_or_search_pmids(
        gene_pair
    )

    new_result = fn.analyze_gene_pair(
        gene_pair,
        candidate_pmids,
    )

    # Adds the new gene without changing existing genes.
    all_results[gene_pair.approved_symbol] = new_result

    # Save after every completed gene.
    save_all_results_json(all_results)

    print(
        f"Saved {gene_pair.approved_symbol}"
    )
    print(
        "Document cache retained at: "
        f"{gene_pair.document_cache_dir}"
    )

Analyzing ABCD1 -> ABCD1...
ABCD1 page 1: 10 new PMIDs (10 total)
ABCD1 page 2: 10 new PMIDs (20 total)
ABCD1 page 3: 10 new PMIDs (30 total)
ABCD1 page 4: 10 new PMIDs (40 total)
ABCD1 page 5: 10 new PMIDs (50 total)
ABCD1 page 6: 10 new PMIDs (60 total)
ABCD1 page 7: 10 new PMIDs (70 total)
ABCD1 page 8: 10 new PMIDs (80 total)
ABCD1 page 9: 10 new PMIDs (90 total)
ABCD1 page 10: 10 new PMIDs (100 total)
ABCD1 page 11: 10 new PMIDs (110 total)
ABCD1 page 12: 10 new PMIDs (120 total)
ABCD1 page 13: 10 new PMIDs (130 total)
ABCD1 page 14: 10 new PMIDs (140 total)
ABCD1 page 15: 10 new PMIDs (150 total)
ABCD1 page 16: 10 new PMIDs (160 total)
ABCD1 page 17: 10 new PMIDs (170 total)
ABCD1 page 18: 10 new PMIDs (180 total)
ABCD1 page 19: 10 new PMIDs (190 total)
ABCD1 page 20: 10 new PMIDs (200 total)
ABCD1 page 21: 10 new PMIDs (210 total)
ABCD1 page 22: 10 new PMIDs (220 total)
ABCD1 page 23: 10 new PMIDs (230 total)
ABCD1 page 24: 10 new PMIDs (240 total)
ABCD1 page 25: 10 new PMIDs (2

## <a id='toc1_4_'></a>[Results](#toc0_)

**Candidate papers for alias:** This is the number of unique PubMed articles returned by the PubTator search for the query "alias". It comes from load_or_search_pmids(pair), which searches PubTator and caches the resulting PMIDs.

**Full-text documents analyzed:** Documents containing at least one passage beyond the title and abstract.

**Papers where PubTator tagged the alias as a gene:** Papers with at least one PubTator annotation where:

- annotation["infons"]["type"] == "gene"
- annotation["text"] == "alias" (case-insensitive)

Each paper is counted only once, even if the alias appears multiple times.

**Papers containing an accepted NCBI Gene, HGNC, Ensembl, or OMIM identifier for the alias:** Papers with one of the following identifier strings in the PubTator text:

- HGNC:XXXX
- ENSG00000XXXXXX
- NCBI Gene XXXX 
- OMIM: XXXX (or equivalent pattern)

This is based on searching all available full-text passages using the regular expressions in make_identifier_patterns().

**Percentage containing an identifier:** numerator / denominator(Papers where PubTator tagged the alias as a gene)

**Counts by identifier namespace**

These are the numerator broken down by identifier type:

- NCBI Gene: papers containing an NCBI Gene identifier for the alias
- HGNC: papers containing an HGNC identifier for the alias
- Ensembl: papers containing an Ensembl identifier for the alias

### <a id='toc1_4_1_'></a>[Ambiguous symbols](#toc0_)

In [9]:
candidate_pmids = fn.load_or_search_pmids(ERBB_EGFR)

results = fn.analyze_gene_pair(
    ERBB_EGFR,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ERBB: 25,467
Full-text documents analyzed: 18,167
Papers where PubTator tagged the exact alias ERBB as a gene: 11,059
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for EGFR: 3
Percentage containing an identifier: 0.03%

Counts by identifier namespace:
Papers containing an accepted identifier for EGFR: 3
Identifier namespaces searched: NCBI Gene, HGNC, Ensembl
Percentage containing an identifier: 0.03%

Counts by identifier namespace:
  NCBI Gene: 1 papers
  HGNC: 0 papers
  Ensembl: 2 papers

Identifier locations (PMIDs):
  fig_caption: 32210363
  paragraph: 26886748
  table: 33232279


In [10]:
results["papers_with_identifier_in_text"]

{'26886748', '32210363', '33232279'}

In [11]:
candidate_pmids = fn.load_or_search_pmids(ASP_TMPRSS11D)

results = fn.analyze_gene_pair(
    ASP_TMPRSS11D,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Full-text documents analyzed: 2,867
Papers where PubTator tagged the exact alias ASP as a gene: 621
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for TMPRSS11D: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
Papers containing an accepted identifier for TMPRSS11D: 0
Identifier namespaces searched: NCBI Gene, HGNC, Ensembl
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [12]:
candidate_pmids = fn.load_or_search_pmids(ASP_C3)

results = fn.analyze_gene_pair(
    ASP_C3,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Full-text documents analyzed: 2,867
Papers where PubTator tagged the exact alias ASP as a gene: 621
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for C3: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
Papers containing an accepted identifier for C3: 0
Identifier namespaces searched: NCBI Gene, HGNC, Ensembl
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [13]:
candidate_pmids = fn.load_or_search_pmids(ASP_ASPM)

results = fn.analyze_gene_pair(
    ASP_ASPM,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Full-text documents analyzed: 2,867
Papers where PubTator tagged the exact alias ASP as a gene: 621
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for ASPM: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
Papers containing an accepted identifier for ASPM: 0
Identifier namespaces searched: NCBI Gene, HGNC, Ensembl
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [14]:
candidate_pmids = fn.load_or_search_pmids(ASP_A1CF)

results = fn.analyze_gene_pair(
    ASP_A1CF,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Full-text documents analyzed: 2,867
Papers where PubTator tagged the exact alias ASP as a gene: 621
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for A1CF: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
Papers containing an accepted identifier for A1CF: 0
Identifier namespaces searched: NCBI Gene, HGNC, Ensembl
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [15]:
candidate_pmids = fn.load_or_search_pmids(ASP_ASIP)

results = fn.analyze_gene_pair(
    ASP_ASIP,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Full-text documents analyzed: 2,867
Papers where PubTator tagged the exact alias ASP as a gene: 621
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for ASIP: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
Papers containing an accepted identifier for ASIP: 0
Identifier namespaces searched: NCBI Gene, HGNC, Ensembl
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [16]:
candidate_pmids = fn.load_or_search_pmids(ASP_ASPA)

results = fn.analyze_gene_pair(
    ASP_ASPA,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Full-text documents analyzed: 2,867
Papers where PubTator tagged the exact alias ASP as a gene: 621
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for ASPA: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
Papers containing an accepted identifier for ASPA: 0
Identifier namespaces searched: NCBI Gene, HGNC, Ensembl
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [17]:
candidate_pmids = fn.load_or_search_pmids(ASP_ATG5)

results = fn.analyze_gene_pair(
    ASP_ATG5,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Full-text documents analyzed: 2,867
Papers where PubTator tagged the exact alias ASP as a gene: 621
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for ATG5: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
Papers containing an accepted identifier for ATG5: 0
Identifier namespaces searched: NCBI Gene, HGNC, Ensembl
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


In [18]:
candidate_pmids = fn.load_or_search_pmids(ASP_ROPN1L)

results = fn.analyze_gene_pair(
    ASP_ROPN1L,
    candidate_pmids,
)

fn.print_analysis_results(results)

Candidate papers for ASP: 6,549
Full-text documents analyzed: 2,867
Papers where PubTator tagged the exact alias ASP as a gene: 621
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for ROPN1L: 0
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
Papers containing an accepted identifier for ROPN1L: 0
Identifier namespaces searched: NCBI Gene, HGNC, Ensembl
Percentage containing an identifier: 0.00%

Counts by identifier namespace:
  NCBI Gene: 0 papers
  HGNC: 0 papers
  Ensembl: 0 papers

Identifier locations (PMIDs):


Pubtator3 normalizes ASP to different genes, there are just no gene identifiers used

In [19]:
papers_by_pubtator_id = defaultdict(set)
annotations_by_pubtator_id = defaultdict(int)
papers_with_no_id = set()

for document in fn.fetch_documents(ASP_ROPN1L, candidate_pmids):
    pmid = str(
        document.get("pmid")
        or document.get("id")
        or ""
    ).strip()

    if not pmid:
        continue

    for passage in document.get("passages", []):
        for annotation in passage.get("annotations", []):
            infons = annotation.get("infons", {})

            entity_type = (
                str(infons.get("type", ""))
                .strip()
                .casefold()
            )

            mention = (
                str(annotation.get("text", ""))
                .strip()
            )

            if (
                entity_type == "gene"
                and mention.casefold() == ASP_ROPN1L.alias.casefold()
            ):
                identifier = infons.get("identifier")

                if identifier:
                    identifier = str(identifier).strip()

                    papers_by_pubtator_id[identifier].add(pmid)
                    annotations_by_pubtator_id[identifier] += 1
                else:
                    papers_with_no_id.add(pmid)

print(f"PubTator normalization results for {ASP_ROPN1L.alias}:\n")

for identifier in sorted(
    papers_by_pubtator_id,
    key=lambda x: len(papers_by_pubtator_id[x]),
    reverse=True,
):
    print(
        f"{identifier}: "
        f"{len(papers_by_pubtator_id[identifier]):,} papers, "
        f"{annotations_by_pubtator_id[identifier]:,} annotations"
    )

print(
    f"\nNo PubTator identifier: "
    f"{len(papers_with_no_id):,} papers"
)

PubTator normalization results for ASP:

259266: 726 papers, 25,529 annotations
2200: 12 papers, 286 annotations
509432: 10 papers, 236 annotations
374569: 4 papers, 94 annotations
434: 3 papers, 64 annotations
492296: 2 papers, 36 annotations
112935892: 1 papers, 3 annotations
104816: 1 papers, 24 annotations
492297: 1 papers, 10 annotations
42946: 1 papers, 1 annotations
100127221: 1 papers, 1 annotations

No PubTator identifier: 0 papers


In [20]:
identifiers_by_pmid = defaultdict(set)

for identifier, pmids in papers_by_pubtator_id.items():
    for pmid in pmids:
        identifiers_by_pmid[pmid].add(identifier)

papers_with_multiple_ids = {
    pmid: identifiers
    for pmid, identifiers in identifiers_by_pmid.items()
    if len(identifiers) > 1
}

print(
    f"Papers where ASP maps to multiple identifiers: "
    f"{len(papers_with_multiple_ids):,}"
)

for pmid, identifiers in sorted(
    papers_with_multiple_ids.items(),
    key=lambda item: len(item[1]),
    reverse=True,
):
    print(f"PMID {pmid}: {sorted(identifiers)}")

Papers where ASP maps to multiple identifiers: 2
PMID 39697544: ['259266', '509432']
PMID 37836499: ['434', '509432']


- PMID 39697544 is about Asparagus officinalis L. (ASP) and it is normalized to ASPM (GENE ID:259266) and ASPA (GENE ID:509432) within the same paper.

- PMID 37836499 is about Agouti signaling protein (ASP) and is normalized to ASIP (GENE ID:434). I could not find the instance where ASP was normalized to ASPA (GENE ID:509432) in this paper.

### <a id='toc1_4_2_'></a>[ACMG gene set](#toc0_)

In [34]:
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [40]:
summary_rows = []

for symbol, result in all_results.items():
    pair = result["pair"]

    row = {
        "approved_symbol": symbol,
        "alias": pair.alias,
        "HGNC_ID": pair.hgnc_id,
        "NCBI_Gene_ID": pair.ncbi_gene_id,
        "Ensembl_Gene_ID": pair.ensembl_gene_id,
        "OMIM_ID": pair.omim_id,
        "candidate_papers":
            result["candidate_papers"],
        "full_text_documents":
            result["processed_documents"],
        "papers_with_alias_gene_annotation":
            result["denominator"],
        "papers_with_identifier":
            result["numerator"],
        "percentage_with_identifier":
            result["percentage"],
    }

    for namespace, pmids in (
        result["papers_by_namespace"].items()
    ):
        row[f"{namespace}_paper_count"] = len(pmids)

    summary_rows.append(row)

summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values(["alias", "approved_symbol"])
    .reset_index(drop=True)
)

summary_df.to_csv(
    "output/all_results_summary.csv",
    index=False,
)

summary_df

,approved_symbol,alias,HGNC_ID,NCBI_Gene_ID,Ensembl_Gene_ID,OMIM_ID,candidate_papers,full_text_documents,papers_with_alias_gene_annotation,papers_with_identifier,percentage_with_identifier,NCBI Gene_paper_count,HGNC_paper_count,Ensembl_paper_count,OMIM_paper_count
0,ABCD1,ABCD1,HGNC:61,215,ENSG00000101986,300371,1843,1385,1119,17,1.519214,1,2,2,12
1,ACTA2,ACTA2,HGNC:130,59,ENSG00000107796,102620,12819,11661,10553,36,0.341135,4,0,15,19
2,ACTC1,ACTC1,HGNC:143,70,ENSG00000159251,102540,2589,2410,1972,20,1.014199,1,0,8,12
3,ACVRL1,ACVRL1,HGNC:175,94,ENSG00000139567,601284,2010,1687,1436,36,2.506964,1,1,5,30
4,APC,APC,HGNC:583,324,ENSG00000134982,611731,20586,13966,5235,16,0.305635,1,0,2,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,TSC2,TSC2,HGNC:12363,7249,ENSG00000103197,191092,8637,6307,5743,19,0.330838,0,1,2,16
80,TTN,TTN,HGNC:12403,7273,ENSG00000155657,188840,4096,3139,2061,36,1.746725,0,4,13,20
81,TTR,TTR,HGNC:12405,7276,ENSG00000118271,176300,17820,12836,6370,29,0.455259,2,5,7,15
82,VHL,VHL,HGNC:12687,7428,ENSG00000134086,608537,20157,15222,11544,26,0.225225,2,1,5,19


In [41]:
location_rows = []

for symbol, result in all_results.items():
    pair = result["pair"]
    locations = result["papers_by_section"]

    if not locations:
        location_rows.append({
            "approved_symbol": symbol,
            "alias": pair.alias,
            "identifier_location": None,
            "paper_count": 0,
            "PMIDs": "",
        })

        continue

    for section, pmids in locations.items():
        sorted_pmids = sorted(
            (str(pmid) for pmid in pmids),
            key=int,
        )

        location_rows.append({
            "approved_symbol": symbol,
            "alias": pair.alias,
            "identifier_location": section,
            "paper_count": len(pmids),
            "PMIDs": ", ".join(sorted_pmids),
        })

identifier_locations_df = (
    pd.DataFrame(location_rows)
    .sort_values(
        [
            "alias",
            "approved_symbol",
            "identifier_location",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

identifier_locations_df.to_csv(
    OUTPUT_DIR
    / "identifier_locations_summary.csv",
    index=False,
)

identifier_locations_df

,approved_symbol,alias,identifier_location,paper_count,PMIDs
0,ABCD1,ABCD1,paragraph,14,"22447153, 26607867, 29794777, 30646031, 309685..."
1,ABCD1,ABCD1,table,3,"32704519, 32794647, 33665191"
2,ACTA2,ACTA2,fig_caption,1,27935821
3,ACTA2,ACTA2,paragraph,24,"19468071, 22187657, 24039846, 25254113, 269344..."
4,ACTA2,ACTA2,table,10,"21360310, 25260786, 27050376, 28123539, 296262..."
...,...,...,...,...,...
240,VHL,VHL,table,5,"22761941, 22788692, 24612714, 33574476, 35406801"
241,VHL,VHL,table_foot,1,33649982
242,WT1,WT1,abstract,1,31013750
243,WT1,WT1,paragraph,32,"19417065, 19562370, 26420286, 27108798, 296189..."


In [39]:
fn.print_analysis_results(all_results["ABCD1"])

Candidate papers for ABCD1: 1,843
Full-text documents analyzed: 1,385
Papers where PubTator tagged the exact alias ABCD1 as a gene: 1,119
Papers containing an accepted NCBI Gene, HGNC, or Ensembl identifier for ABCD1: 17
Percentage containing an identifier: 1.52%

Counts by identifier namespace:
Papers containing an accepted identifier for ABCD1: 17
Identifier namespaces searched: NCBI Gene, HGNC, Ensembl, OMIM
Percentage containing an identifier: 1.52%

Counts by identifier namespace:
  NCBI Gene: 1 papers
  HGNC: 2 papers
  Ensembl: 2 papers
  OMIM: 12 papers

Identifier locations (PMIDs):
  paragraph: 22447153, 26607867, 29794777, 30646031, 30968598, 31777199, 32003821, 32047678, 32730690, 35791118, 35996156, 36527290, 37641794, 40021620
  table: 32704519, 32794647, 33665191
